In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", None)

In [2]:
project_root = Path.cwd().parent

metadata_dir = (
    project_root
    / "data"
    / "raw"
    / "abo-listings"
    / "listings"
    / "metadata"
)

metadata_files = sorted(metadata_dir.glob("listings_*.json.gz"))

print(f"Metadata files found: {len(metadata_files)}")

Metadata files found: 16


In [3]:
all_dfs = []

for file in metadata_files:
    df = pd.read_json(file, lines=True)
    all_dfs.append(df)

    print(f"{file.name}  -->  {len(df):,} rows")

full_df = pd.concat(all_dfs, ignore_index=True)

print("\n--------------------------------")
print(f"Total Products : {len(full_df):,}")
print(f"Total Columns  : {len(full_df.columns)}")

listings_0.json.gz  -->  9,232 rows
listings_1.json.gz  -->  9,232 rows
listings_2.json.gz  -->  9,232 rows
listings_3.json.gz  -->  9,232 rows
listings_4.json.gz  -->  9,232 rows
listings_5.json.gz  -->  9,232 rows
listings_6.json.gz  -->  9,232 rows
listings_7.json.gz  -->  9,232 rows
listings_8.json.gz  -->  9,232 rows
listings_9.json.gz  -->  9,232 rows
listings_a.json.gz  -->  9,232 rows
listings_b.json.gz  -->  9,232 rows
listings_c.json.gz  -->  9,232 rows
listings_d.json.gz  -->  9,232 rows
listings_e.json.gz  -->  9,232 rows
listings_f.json.gz  -->  9,222 rows

--------------------------------
Total Products : 147,702
Total Columns  : 28


## Step 3 - Extract Required Columns and Clean Metadata

In [4]:
def extract_text(value):
    """
    Extract the first text value from ABO metadata fields.
    """
    if isinstance(value, list) and len(value) > 0:
        return value[0].get("value")
    return None


clean_df = pd.DataFrame({
    "item_id": full_df["item_id"],
    "title": full_df["item_name"].apply(extract_text),
    "product_type": full_df["product_type"].apply(extract_text),
    "brand": full_df["brand"].apply(extract_text),
    "color": full_df["color"].apply(extract_text)
})

# Remove rows with missing title or product type
clean_df = clean_df.dropna(subset=["title", "product_type"])

print("Dataset Shape:", clean_df.shape)
print()

clean_df.head()

Dataset Shape: (147702, 5)



,item_id,title,product_type,brand,color
0,B06X9STHNG,Amazon-merk - vinden. Dames Leder Gesloten Tee...,SHOES,find.,Veelkleurig Vrouw Blauw
1,B07P8ML82R,"22"" Bottom Mount Drawer Slides, White Powder C...",HARDWARE,AmazonBasics,White Powder Coat
2,B07H9GMYXS,"AmazonBasics PETG 3D Printer Filament, 1.75mm,...",MECHANICAL_COMPONENTS,AmazonBasics,Translucent Yellow
3,B07CTPR73M,"Stone & Beam Stone Brown Swatch, 25020039-01",SOFA,Stone & Beam,Stone Brown
4,B01MTEI8M6,The Fix Amazon Brand Women's French Floral Emb...,SHOES,The Fix,Havana Tan


## Step 4 - Analyze Product Types Distribution

In [5]:
product_counts = (
    clean_df["product_type"]
    .value_counts()
    .reset_index()
)

product_counts.columns = ["product_type", "count"]

print("Total Unique Product Types:", len(product_counts))
print()

product_counts.head(100)

Total Unique Product Types: 576



,product_type,count
0,CELLULAR_PHONE_CASE,64853
1,SHOES,12965
2,GROCERY,6546
3,HOME,5264
4,HOME_BED_AND_BATH,3082
...,...,...
95,FURNITURE,123
96,CLOTHES_HANGER,122
97,DAIRY_BASED_DRINK,114
98,STORAGE_BAG,114


## Step 5 - Explore Product Types for Target Categories

In [6]:
keywords = {
    "Footwear": ["SHOE", "BOOT", "SANDAL", "SLIPPER", "SNEAKER", "TRAINER", "HEEL"],
    "Bags": ["BAG", "BACKPACK", "HANDBAG", "LUGGAGE", "SUITCASE", "PURSE", "TOTE"],
    "Watches": ["WATCH", "CLOCK"],
    "Furniture": ["SOFA", "CHAIR", "TABLE", "DESK", "BED", "CABINET",
                  "FURNITURE", "BOOKCASE", "BOOKSHELF", "STOOL",
                  "BENCH", "SHELF", "DRAWER"],
    "Kitchenware": ["KITCHEN", "PLATE", "BOWL", "CUP", "MUG",
                    "GLASS", "PAN", "POT", "KNIFE", "SPOON",
                    "FORK", "COOK", "DINING", "BAKE", "TRAY"],
    "Electronics": ["PHONE", "CELLULAR", "HEADPHONE", "EARPHONE",
                    "CHARGER", "CABLE", "USB", "POWER",
                    "KEYBOARD", "MOUSE", "SPEAKER",
                    "MONITOR", "LAPTOP", "COMPUTER"]
}

for category, words in keywords.items():

    print("=" * 70)
    print(category)
    print("=" * 70)

    temp = product_counts[
        product_counts["product_type"].str.contains(
            "|".join(words),
            case=False,
            na=False
        )
    ]

    display(temp)

Footwear


,product_type,count
1,SHOES,12965
7,BOOT,2009
8,SANDAL,1845
197,TECHNICAL_SPORT_SHOE,37
250,SHOE_INSERT,23
271,STEERING_WHEEL_COVER,19
291,WHEEL,16
512,WHEEL_CUTTER,2


Bags


,product_type,count
21,HANDBAG,792
34,SUITCASE,429
54,BACKPACK,258
57,LUGGAGE,243
74,WASTE_BAG,165
98,STORAGE_BAG,114
135,BEAN_BAG_CHAIR,73
180,TOTE_BAG,44
217,CAMERA_BAGS_AND_CASES,30
227,CARRYING_CASE_OR_BAG,29


Watches


...

,product_type,count
4,HOME_BED_AND_BATH,3082
5,HOME_FURNITURE_AND_DECOR,2255
6,CHAIR,2100
13,SOFA,1199
18,TABLE,936
...,...,...
443,PORTABLE_STOVE,4
453,PORTABLE_TOOL_BOX,4
458,AMAZON_TABLET_ACCESSORY,4
462,SPORT_TABLE_GAME,3


Kitchenware


,product_type,count
23,KITCHEN,706
39,DRINKING_CUP,380
66,PANTRY,201
70,SAUTE_FRY_PAN,168
141,DISHWARE_PLATE,71
...,...,...
499,COOKIE_CUTTER,2
507,ICE_CUBE_TRAY,2
513,KNIFE_BLOCK_SET,2
570,COOKING_OVEN,1


Electronics


,product_type,count
0,CELLULAR_PHONE_CASE,64853
65,COMPUTER_ADD_ON,204
77,HEADPHONES,156
100,COMPUTER_COMPONENT,112
108,SPEAKERS,103
...,...,...
536,COMPUTER_COOLING_DEVICE,1
539,WIRELESS_LOCKED_PHONE,1
542,MONITOR,1
554,BLOOD_PRESSURE_MONITOR,1


## Step 6 - Define Production Category Mapping

In [10]:
CATEGORY_MAPPING = {

    "Footwear": [
        "SHOES",
        "BOOT",
        "SANDAL",
        "TECHNICAL_SPORT_SHOE"
    ],

    "Furniture": [
        "SOFA",
        "CHAIR",
        "TABLE",
        "HOME_FURNITURE_AND_DECOR",
        "HOME_BED_AND_BATH",
        "FURNITURE",
        "DESK",
        "BENCH",
        "STOOL",
        "BOOKCASE",
        "BOOKSHELF",
        "CABINET"
    ],

    "Electronics_Accessories": [
        "CELLULAR_PHONE_CASE",
        "HEADPHONES",
        "SPEAKERS",
        "COMPUTER_COMPONENT",
        "COMPUTER_ADD_ON",
        "CE_CARRYING_CASE_OR_BAG",
        "AMAZON_TABLET_ACCESSORY",
        "CELLULAR_PHONE"
    ],

    "Fashion_Travel": [
        "HANDBAG",
        "BACKPACK",
        "SUITCASE",
        "LUGGAGE",
        "TOTE_BAG",
        "CAMERA_BAGS_AND_CASES",
        "CARRYING_CASE_OR_BAG",
        "BAG"
    ],

    "Home_Kitchen": [
        "KITCHEN",
        "DRINKING_CUP",
        "PANTRY",
        "SAUTE_FRY_PAN",
        "DISHWARE_PLATE",
        "MEASURING_CUP",
        "BAKING_CUP",
        "KITCHEN_KNIFE",
        "KITCHEN_TOOLS",
        "BAKEWARE",
        "COOKING_OVEN",
        "ICE_CUBE_TRAY",
        "KNIFE_BLOCK_SET"
    ],

    "Hardware_HomeImprovement": [
        "HARDWARE",
        "HARDWARE_HANDLE",
        "TOOL",
        "TOOLS",
        "HOME",
        "HOME_IMPROVEMENT"
    ]
}

## Step 7 - Assign Final Categories Using Configuration

In [11]:
def assign_category(product_type):
    """
    Map an ABO product_type to one of the
    production categories.
    """

    if pd.isna(product_type):
        return None

    product_type = str(product_type).strip().upper()

    for category, product_list in CATEGORY_MAPPING.items():
        if product_type in product_list:
            return category

    return None


# Create a copy
production_df = clean_df.copy()

# Assign categories
production_df["category"] = production_df["product_type"].apply(assign_category)

# Keep only mapped products
production_df = production_df.dropna(subset=["category"])

print("Production Dataset Shape:", production_df.shape)
print()

print("Category Distribution")
print("-" * 40)
print(production_df["category"].value_counts())

Production Dataset Shape: (102683, 6)

Category Distribution
----------------------------------------
category
Electronics_Accessories     65450
Footwear                    16856
Furniture                   10209
Hardware_HomeImprovement     6762
Fashion_Travel               1839
Home_Kitchen                 1567
Name: count, dtype: int64


## Step 8 - Explore Additional Product Types

In [9]:
def show_matching(keyword):
    result = product_counts[
        product_counts["product_type"].str.contains(
            keyword,
            case=False,
            na=False
        )
    ]

    print(f"\nKeyword: {keyword}")
    display(result)


keywords = [
    "BAG",
    "CASE",
    "LUGGAGE",
    "KITCHEN",
    "PLATE",
    "CUP",
    "PAN",
    "POT",
    "SPOON",
    "KNIFE",
    "FORK",
    "TRAY",
    "HARDWARE",
    "TOOL",
    "HOME",
    "STORAGE"
]

for word in keywords:
    show_matching(word)


Keyword: BAG


,product_type,count
21,HANDBAG,792
74,WASTE_BAG,165
98,STORAGE_BAG,114
135,BEAN_BAG_CHAIR,73
180,TOTE_BAG,44
217,CAMERA_BAGS_AND_CASES,30
227,CARRYING_CASE_OR_BAG,29
232,FOOD_STORAGE_BAG,28
283,CE_CARRYING_CASE_OR_BAG,17
311,BAG,14



Keyword: CASE


,product_type,count
0,CELLULAR_PHONE_CASE,64853
34,SUITCASE,429
151,COSMETIC_CASE,65
217,CAMERA_BAGS_AND_CASES,30
227,CARRYING_CASE_OR_BAG,29
283,CE_CARRYING_CASE_OR_BAG,17



Keyword: LUGGAGE


...

,product_type,count
37,STOOL_SEATING,393
40,TOOLS,379
366,MULTITOOL,8
449,KITCHEN_TOOLS,4
453,PORTABLE_TOOL_BOX,4
555,VEHICLE_SCAN_TOOL,1



Keyword: HOME


,product_type,count
3,HOME,5264
4,HOME_BED_AND_BATH,3082
5,HOME_FURNITURE_AND_DECOR,2255
85,HOME_LIGHTING_AND_LAMPS,146
101,HOME_MIRROR,109
114,MAJOR_HOME_APPLIANCES,96
164,HOME_LIGHTING_ACCESSORY,56
331,ABIS_HOME_IMPROVEMENT,12
515,SMALL_HOME_APPLIANCES,2
544,HOME_ORGANIZERS_AND_STORAGE,1



Keyword: STORAGE


,product_type,count
60,STORAGE_HOOK,236
67,STORAGE_BINDER,188
98,STORAGE_BAG,114
110,STORAGE_BOX,102
178,COMPUTER_DRIVE_OR_STORAGE,45
232,FOOD_STORAGE_BAG,28
238,JEWELRY_STORAGE,27
241,STORAGE_RACK,26
262,STORAGE_DRAWER,21
410,MEDIA_STORAGE,6
